# 07 — Final results on the test set
Validation'da en iyi kurulum seçilir ve **test seti sadece bir kez** değerlendirilir. Bu sayı modelin hiç görmediği veride beklenen performanstır.

In [ ]:
# On Colab: mount Drive and work from the project's notebooks folder
import os, sys
if "google.colab" in sys.modules and not os.getcwd().endswith("notebooks"):
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/bone-age/notebooks
%run 00_common.ipynb

In [ ]:
experiments = pd.read_json(OUTPUT_DIR / "experiments.json")
best = experiments.iloc[0]
print(f"Selected on validation: {best['setup']} (val MAE {best['val_mae']:.2f})")
res, metrics = evaluate(best["runs"], "test", bool(best["tta"]))
out_dir = OUTPUT_DIR / "test_results"
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / "predictions.csv", index=False)
(out_dir / "metrics.json").write_text(json.dumps({"setup": best["setup"], **metrics}, indent=2))
print(f"TEST  MAE {metrics['mae']:.2f} mo | RMSE {metrics['rmse']:.2f} mo | R2 {metrics['r2']:.4f}")

In [ ]:
base = pd.read_csv(OUTPUT_DIR / "baselines.csv").set_index("split")
pd.DataFrame({"Test MAE (months)": {
    "Baseline: train mean": base.loc["test", "mean_baseline_mae"],
    "Baseline: mean per sex": base.loc["test", "sex_mean_baseline_mae"],
    f"Ours: {best['setup']}": metrics["mae"],
    "RSNA 2017 winner (official test set)": 4.27,
}}).round(2)

In [ ]:
plot_results(res, out_dir)

In [ ]:
by_age = res.groupby("age_group", observed=True)["abs_error"].agg(MAE="mean", n="size")
by_sex = res.groupby(res["male"].map({True: "male", False: "female"}))["abs_error"].agg(MAE="mean", n="size")
display(by_age.round(2), by_sex.round(2))

In [ ]:
worst = show_worst(res, out_dir)
worst[["id", "male", "boneage", "pred", "error"]].round(1)